In [ ]:
# ============================================================
# 1. Save backdoor information once.
# ============================================================
import torch
import torchvision
import random
from pathlib import Path

def create_backdoor_info(
    spuriosity,
    save_dir='/home/dataset/mnist_backdoor',
    seed=42
):
    """Save only the backdoor class information."""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    for train in [True, False]:
        split = 'train' if train else 'test'

        mnist = torchvision.datasets.MNIST(
            root='/home/dataset/mnist', train=train, download=False
        )

        rng = random.Random(seed)
        bd_classes = []

        for idx in range(len(mnist)):
            _, label = mnist[idx]

            if spuriosity is not None:
                if rng.random() < spuriosity:
                    bd_class = label
                else:
                    bd_class = (label + rng.randint(1, 9)) % 10
            else:
                bd_class = rng.randint(0, 9)

            bd_classes.append(bd_class)

        # Save
        save_path = save_dir / f'{split}_spur{spuriosity}_seed{seed}.pt'
        torch.save(torch.tensor(bd_classes, dtype=torch.long), save_path)
        print(f"Saved {len(bd_classes)} entries to {save_path}")

# Run
create_backdoor_info(spuriosity=1.0, seed=42)
create_backdoor_info(spuriosity=0.7, seed=42)
create_backdoor_info(spuriosity=None, seed=42)